# Topic 30 — Build an MLP
### Theory → architecture → full MLP on a real multi-class dataset → training loop with train/val tracking.

A **Multilayer Perceptron (MLP)** is simply several fully-connected (`Linear`) layers stacked with
activations in between — the general version of the 1-hidden-layer network from Topics 27-29.

```text
Input -> Linear -> ReLU -> Linear -> ReLU -> Linear -> Output
```

This notebook builds a proper multi-layer MLP on a real dataset (handwritten digits, 10 classes)
with a full train/validation loop, tracking BOTH loss and accuracy every epoch — the standard
structure you'll reuse for every deep learning experiment from here on, including your DIP
miniproject if you go the neural-network route there.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler

device = "cuda" if torch.cuda.is_available() else "cpu"
print("using device:", device)
torch.manual_seed(42)

## 1. Load and prepare data

Reusing the digits dataset from Topic 19: 1797 samples, 64 features (8x8 pixel images), 10 classes
(digits 0-9). Standard scaling (Topic 15) helps MLPs train faster and more stably.

In [ ]:
digits = load_digits()
X, y = digits.data, digits.target

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)   # fit on full data here for simplicity; in a real project
                                       # fit ONLY on train (Topic 5/15) before this split

X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.long)   # class INDICES (0-9), not one-hot, for CrossEntropyLoss

# Peek at a few digit images
fig, axes = plt.subplots(1, 5, figsize=(10, 2))
for i, ax in enumerate(axes):
    ax.imshow(digits.images[i], cmap="gray")
    ax.set_title(f"label: {digits.target[i]}")
    ax.axis("off")
plt.show()

## 2. Dataset, train/val split, DataLoaders

In [ ]:
class DigitsDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

full_dataset = DigitsDataset(X_tensor, y_tensor)
n_train = int(0.8 * len(full_dataset))
n_val = len(full_dataset) - n_train
train_dataset, val_dataset = random_split(full_dataset, [n_train, n_val],
                                           generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print("train samples:", len(train_dataset), " val samples:", len(val_dataset))

## 3. The MLP architecture

Multiple hidden layers, going 64 -> 64 -> 32 -> 10. The final layer has 10 outputs (one score per
digit class) and NO activation applied here — `nn.CrossEntropyLoss` applies softmax internally,
so raw scores ("logits") go in directly.

In [ ]:
class MLP(nn.Module):
    def __init__(self, n_input=64, n_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_input, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, n_classes),   # raw logits -- CrossEntropyLoss handles softmax internally
        )

    def forward(self, x):
        return self.net(x)

model = MLP().to(device)
print(model)
n_params = sum(p.numel() for p in model.parameters())
print("\ntotal parameters:", n_params)

## 4. Training loop with train + validation tracking every epoch

This is the standard loop structure you'll reuse for CNNs, RNNs, and beyond.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def compute_accuracy(loader, model):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            predicted = outputs.argmax(dim=1)     # class with highest score
            correct += (predicted == y_batch).sum().item()
            total += y_batch.size(0)
    return correct / total

n_epochs = 30
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(n_epochs):
    model.train()
    train_losses = []
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    model.eval()
    val_losses = []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            val_losses.append(criterion(outputs, y_batch).item())

    history["train_loss"].append(np.mean(train_losses))
    history["val_loss"].append(np.mean(val_losses))
    history["train_acc"].append(compute_accuracy(train_loader, model))
    history["val_acc"].append(compute_accuracy(val_loader, model))

    if (epoch + 1) % 5 == 0:
        print(f"epoch {epoch+1:>2}: train_loss={history['train_loss'][-1]:.3f}  "
              f"val_loss={history['val_loss'][-1]:.3f}  "
              f"train_acc={history['train_acc'][-1]:.3f}  val_acc={history['val_acc'][-1]:.3f}")

## 5. Plotting the training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend()

axes[1].plot(history["train_acc"], label="train")
axes[1].plot(history["val_acc"], label="val")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout()
plt.show()
print(f"final val accuracy: {history['val_acc'][-1]:.3f}")
# Watch the train/val gap here -- this is the SAME overfitting signature from Topic 5, now
# appearing in a neural network instead of a polynomial regression.

## 6. Inspecting a few predictions

In [ ]:
model.eval()
sample_X, sample_y = next(iter(val_loader))
with torch.no_grad():
    sample_preds = model(sample_X.to(device)).argmax(dim=1).cpu()

fig, axes = plt.subplots(1, 6, figsize=(12, 2))
for i, ax in enumerate(axes):
    img = sample_X[i].numpy().reshape(8, 8)
    ax.imshow(img, cmap="gray")
    color = "green" if sample_preds[i] == sample_y[i] else "red"
    ax.set_title(f"pred:{sample_preds[i].item()} true:{sample_y[i].item()}", color=color, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Exercise

In [ ]:
# --- Try it yourself ---
# 1. Add a 4th hidden layer (nn.Linear(32, 16), nn.ReLU()) before the final output layer.
# 2. Change the learning rate to 0.0001 and 0.01 -- compare the loss curves' shapes.
# 3. Train for 100 epochs instead of 30 -- does the train/val gap widen (early sign of overfitting,
#    a preview of Topic 31)?
# 4. Swap CrossEntropyLoss's model output for MSE against one-hot labels instead (you'll need to
#    one-hot encode y and change the final layer to include softmax) -- notice this is MORE work
#    and less numerically stable, which is exactly why CrossEntropyLoss exists as a combined tool.

---
### Next up: **Topic 31 — Overfitting in Deep Learning** (dropout, regularization, early stopping, batch norm).

Say "next" when you're ready.